In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [100]:
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')

In [34]:
train_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0.0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0.0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0.0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0.0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0.0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [35]:
test_df.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [39]:
len(train_df.columns), train_df.columns

(11,
 Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
        'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
        'Weather'],
       dtype='object'))

In [40]:
len(test_df.columns), test_df.columns

(10,
 Index(['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes',
        'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'],
       dtype='object'))

In [41]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  float64
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(3), int64(3), object(5)
memory usage: 6.5+ MB


In [42]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


In [101]:
train_df.isnull().sum(), test_df.isnull().sum()

(Index               0
 geohash             0
 day                 0
 timestamp           0
 demand              0
 RoadType          600
 NumberofLanes       0
 LargeVehicles       0
 Landmarks           0
 Temperature      2495
 Weather           797
 dtype: int64,
 Index               0
 geohash             0
 day                 0
 timestamp           0
 RoadType          324
 NumberofLanes       0
 LargeVehicles       0
 Landmarks           0
 Temperature      1349
 Weather           431
 dtype: int64)

In [102]:
for col in ['RoadType', 'Temperature', 'Weather']:
    print(f'{col} unique values: {train_df[col].unique()}')

RoadType unique values: [nan 'Residential' 'Street' 'Highway']
Temperature unique values: [        nan 31.1045646  25.91926732 ... 19.67885975 22.57395805
  1.32203433]
Weather unique values: [nan 'Sunny' 'Rainy' 'Foggy' 'Snowy']


### Fill Null Value

In [103]:
# number column: Fill missing Temperature with the Median 
train_df['Temperature'] = train_df['Temperature'].fillna(train_df['Temperature'].median())

# categorical columns: Fill missing values with a new category 'Unknown' 
# (Best practice for ML to preserve the fact that data was missing)
train_df['RoadType'] = train_df['RoadType'].fillna('Unknown')
train_df['Weather'] = train_df['Weather'].fillna('Unknown')

# fill categorical columns with the most frequent value:
# train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
# train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])

In [104]:
train_df.isnull().sum(), test_df.isnull().sum()

(Index            0
 geohash          0
 day              0
 timestamp        0
 demand           0
 RoadType         0
 NumberofLanes    0
 LargeVehicles    0
 Landmarks        0
 Temperature      0
 Weather          0
 dtype: int64,
 Index               0
 geohash             0
 day                 0
 timestamp           0
 RoadType          324
 NumberofLanes       0
 LargeVehicles       0
 Landmarks           0
 Temperature      1349
 Weather           431
 dtype: int64)

#### Convert timestamp into meaningfull

In [105]:
train_df["timestamp"].unique()

array([ 0.  ,  0.15,  0.3 ,  0.45,  1.  ,  1.15,  1.3 ,  1.45,  2.  ,
        2.15,  2.3 ,  2.45,  3.  ,  3.15,  3.3 ,  3.45,  4.  ,  4.15,
        4.3 ,  4.45,  5.  ,  5.15,  5.3 ,  5.45,  6.  ,  6.15,  6.3 ,
        6.45,  7.  ,  7.15,  7.3 ,  7.45,  8.  ,  8.15,  8.3 ,  8.45,
        9.  ,  9.15,  9.3 ,  9.45, 10.  , 10.15, 10.3 , 10.45, 11.  ,
       11.15, 11.3 , 11.45, 12.  , 12.15, 12.3 , 12.45, 13.  , 13.15,
       13.3 , 13.45, 14.  , 14.15, 14.3 , 14.45, 15.  , 15.15, 15.3 ,
       15.45, 16.  , 16.15, 16.3 , 16.45, 17.  , 17.15, 17.3 , 17.45,
       18.  , 18.15, 18.3 , 18.45, 19.  , 19.15, 19.3 , 19.45, 20.  ,
       20.15, 20.3 , 20.45, 21.  , 21.15, 21.3 , 21.45, 22.  , 22.15,
       22.3 , 22.45, 23.  , 23.15, 23.3 , 23.45])

In [106]:
train_df.info(), test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  float64
 4   demand         77299 non-null  float64
 5   RoadType       77299 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    77299 non-null  float64
 10  Weather        77299 non-null  object 
dtypes: float64(3), int64(3), object(5)
memory usage: 6.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash    

(None, None)

In [107]:
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])

In [108]:
test_df['timestamp'].unique()

array(['2:15', '2:30', '2:45', '3:0', '3:15', '3:30', '3:45', '4:0',
       '4:15', '4:30', '4:45', '5:0', '5:15', '5:30', '5:45', '6:0',
       '6:15', '6:30', '6:45', '7:0', '7:15', '7:30', '7:45', '8:0',
       '8:15', '8:30', '8:45', '9:0', '9:15', '9:30', '9:45', '10:0',
       '10:15', '10:30', '10:45', '11:0', '11:15', '11:30', '11:45',
       '12:0', '12:15', '12:30', '12:45', '13:0', '13:15', '13:30',
       '13:45'], dtype=object)

In [109]:
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"], format="%H:%M")

In [110]:
train_df['hour'] = train_df['timestamp'].dt.hour
train_df['minute'] = train_df['timestamp'].dt.minute

test_df['hour'] = test_df['timestamp'].dt.hour
test_df['minute'] = test_df['timestamp'].dt.minute

In [111]:
train_df.drop('timestamp', axis=1, inplace=True)
test_df.drop('timestamp', axis=1, inplace=True)

In [112]:
test_df.head(), train_df.head()

(   Index geohash  day     RoadType  NumberofLanes LargeVehicles Landmarks  \
 0      0  qp02z1   49          NaN              1   Not Allowed        No   
 1      1  qp02z9   49  Residential              1   Not Allowed        No   
 2      2  qp02yf   49  Residential              3       Allowed       Yes   
 3      3  qp02z6   49  Residential              2   Not Allowed       Yes   
 4      4  qp02zd   49  Residential              1   Not Allowed        No   
 
    Temperature Weather  hour  minute  
 0          NaN     NaN     2      15  
 1     6.476213   Snowy     2      15  
 2    22.318203   Sunny     2      15  
 3          NaN   Rainy     2      15  
 4    18.266162   Foggy     2      15  ,
    Index geohash  day    demand     RoadType  NumberofLanes LargeVehicles  \
 0      0  qp02z1   48  0.048804      Unknown              1   Not Allowed   
 1      1  qp02zt   48  0.118507  Residential              3       Allowed   
 2      2  qp08bj   48  0.027132  Residential          

In [113]:
train_df.shape, test_df.shape

((77299, 12), (41778, 11))

In [114]:
X = train_df.drop(columns=['Index', 'demand'])
y = train_df['demand']

In [115]:
X.shape, X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   geohash        77299 non-null  object 
 1   day            77299 non-null  int64  
 2   RoadType       77299 non-null  object 
 3   NumberofLanes  77299 non-null  int64  
 4   LargeVehicles  77299 non-null  object 
 5   Landmarks      77299 non-null  object 
 6   Temperature    77299 non-null  float64
 7   Weather        77299 non-null  object 
 8   hour           77299 non-null  int32  
 9   minute         77299 non-null  int32  
dtypes: float64(1), int32(2), int64(2), object(5)
memory usage: 5.3+ MB


((77299, 10), None)

#### Encodeing

In [116]:
# XGBoost requires categorical columns to be explicitly set as 'category' dtype
X.select_dtypes(include=['object']).columns

Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather'], dtype='object')

In [118]:
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

In [119]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   geohash        77299 non-null  category
 1   day            77299 non-null  int64   
 2   RoadType       77299 non-null  category
 3   NumberofLanes  77299 non-null  int64   
 4   LargeVehicles  77299 non-null  category
 5   Landmarks      77299 non-null  category
 6   Temperature    77299 non-null  float64 
 7   Weather        77299 non-null  category
 8   hour           77299 non-null  int32   
 9   minute         77299 non-null  int32   
dtypes: category(5), float64(1), int32(2), int64(2)
memory usage: 2.8 MB


### Train

In [120]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

In [121]:

train_X, valid_X, train_y, valid_y = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [122]:
# Initialize XGBoost Regressor
from tabnanny import verbose


model = XGBRegressor(
    n_estimators=1000,          # Equivalent to iterations in CatBoost
    learning_rate=0.05,
    max_depth=100,
    tree_method='hist',         # REQUIRED for categorical data support
    enable_categorical=True,    # REQUIRED to process the 'category' dtypes
    early_stopping_rounds=50,   # Stops if validation loss doesn't improve
    random_state=42
)

In [123]:
model.fit(
    train_X, train_y,
    eval_set=[(valid_X, valid_y)], # Used for early stopping
    verbose=10                  # Prints progress every 100 trees
)

[0]	validation_0-rmse:0.13597
[10]	validation_0-rmse:0.09016
[20]	validation_0-rmse:0.06734
[30]	validation_0-rmse:0.05773
[40]	validation_0-rmse:0.05453
[50]	validation_0-rmse:0.05386
[60]	validation_0-rmse:0.05396
[70]	validation_0-rmse:0.05427
[80]	validation_0-rmse:0.05464
[90]	validation_0-rmse:0.05493
[100]	validation_0-rmse:0.05517
[103]	validation_0-rmse:0.05522


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,None


In [124]:
pred = model.predict(valid_X)

r2 = r2_score(valid_y, pred)
print(r2)


0.8565485044037183


In [142]:
# save the model for later use
import joblib

joblib.dump(model, 'a_xgb_model.pkl')

['a_xgb_model.pkl']

### problem test submission

In [125]:
test_df.isnull().sum()

Index               0
geohash             0
day                 0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
hour                0
minute              0
dtype: int64

In [126]:
test_df['Temperature'] = test_df['Temperature'].fillna(train_df['Temperature'].median())

test_df['RoadType'] = test_df['RoadType'].fillna('Unknown')
test_df['Weather'] = test_df['Weather'].fillna('Unknown')

# test_df['RoadType'] = test_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
# test_df['Weather'] = test_df['Weather'].fillna(train_df['Weather'].mode()[0])

In [127]:
test_df.isnull().sum()

Index            0
geohash          0
day              0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
hour             0
minute           0
dtype: int64

In [128]:
test_X = test_df.drop(columns=['Index']) 

In [129]:
for col in test_X.select_dtypes(include=['object']).columns:
    test_X[col] = test_X[col].astype('category')

In [130]:
test_pred = model.predict(test_X)
test_X.shape, test_pred.shape

((41778, 10), (41778,))

In [135]:
submit_ans = pd.DataFrame({
    'Index': test_df['Index'],
    "demand": test_pred
})
submit_ans.shape

(41778, 2)

In [141]:
submit_ans.to_csv('submission_a.csv', index=False)

In [140]:
submit_ans.head()

,Index,demand
0,0,0.051093
1,1,0.029437
2,2,0.107750
3,3,0.012888
4,4,0.052069
